# 실습 D: 스마트 질문 라우터

Ch05에서 배운 LCEL 고급 패턴을 종합해서 실제로 활용되는 **스마트 라우팅 파이프라인**을 직접 구축해 볼게요.<br/>
언어 감지, 병렬 분석, 폴백(Fallback), 데이터 보강까지 — 4개 실습을 통해 단계별로 완성해요.

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:
- `RunnableLambda`와 `route()` 패턴으로 동적 라우팅 체인을 구성할 수 있어요
- `RunnableParallel`로 여러 분석 체인을 동시에 실행하고 결과를 합칠 수 있어요
- `with_fallbacks()`로 장애 대응 체인을 설계하고 검증할 수 있어요
- `RunnablePassthrough.assign()`과 `@chain` 데코레이터로 커스텀 파이프라인을 작성할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 Ch05 Advanced LCEL 노트북에서 다룬 `RunnablePassthrough`, `RunnableLambda`, `RunnableParallel`, Routing, Fallbacks, `@chain` 데코레이터, Binding 개념을 이해하고 있어야 해요.

---

아래 다이어그램은 이번 실습에서 구현할 4개 파이프라인 전체 구조를 보여줘요.

```mermaid
graph TD
    IN["입력 텍스트"]:::input

    IN --> EX1["실습 1<br/>다국어 감지 라우터"]:::process
    IN --> EX2["실습 2<br/>병렬 분석 파이프라인"]:::process
    IN --> EX3["실습 3<br/>장애 대응 체인"]:::process
    IN --> EX4["실습 4<br/>맞춤형 데이터 변환"]:::process

    EX1 --> OUT1["번역 결과"]:::output
    EX2 --> OUT2["통합 분석 보고서"]:::output
    EX3 --> OUT3["안정적 응답"]:::output
    EX4 --> OUT4["최종 처리 결과"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 환경 설정

실습에 필요한 라이브러리를 불러오고, API 키를 설정해요.

In [19]:
# ---------------------------------------------------
# API 키 로드 및 환경 설정
# ---------------------------------------------------
from dotenv import load_dotenv

# .env 파일에 저장된 OPENAI_API_KEY를 불러와요
load_dotenv()

True

In [20]:
# ---------------------------------------------------
# LangSmith 추적 설정 (실행 흐름을 추적하고 싶을 때 활성화)
# ---------------------------------------------------
from langchain_teddynote import logging

logging.langsmith("CH05-Practice-D-SmartRouter")

LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954


In [21]:
# ---------------------------------------------------
# 공통 라이브러리 import
# ---------------------------------------------------
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
    chain,
)

# 실습 전반에서 재사용할 기본 모델
# temperature=0: 일관된 출력을 위해 창의성을 최소화해요
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("환경 설정 완료!")

환경 설정 완료!


---

## 실습 1: 다국어 감지 + 번역 라우터

앞서 Routing 노트북에서 주제 분류 후 라우팅하는 방법을 배웠어요.<br/>
이번 실습에서는 입력 텍스트의 **언어를 LLM으로 자동 감지**한 뒤, 감지된 언어에 따라 서로 다른 번역 체인으로 라우팅하는 스마트 라우터를 만들어요.

아래 다이어그램은 다국어 감지 라우터의 동작 흐름을 보여줘요.

```mermaid
graph LR
    A["입력 텍스트"]:::input
    A -->|"언어 감지"| B["언어 감지 체인<br/>(LLM)"]:::process
    B -->|"한국어"| C["한국어 → 영어<br/>번역 체인"]:::process
    B -->|"영어"| D["영어 → 한국어<br/>번역 체인"]:::process
    B -->|"기타"| E["기타 → 한국어<br/>번역 체인"]:::process
    C --> F["번역 결과"]:::output
    D --> F
    E --> F

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

### 실습 1 과제

**과제**: 언어 감지 체인과 3개의 번역 체인을 구성하고, `RunnableLambda`로 감지된 언어에 따라 적절한 번역 체인으로 라우팅하는 전체 파이프라인을 완성하세요.

**요구사항**:
- 언어 감지 체인은 입력 텍스트의 언어를 `한국어`, `영어`, `기타` 중 하나로만 반환해야 해요
- 한국어 입력은 영어로, 영어 입력은 한국어로, 기타 언어는 한국어로 번역해요
- `RunnableLambda`로 라우팅 함수를 래핑해서 전체 체인을 연결해요

**힌트**: `{"language": detect_chain, "text": RunnablePassthrough()}` 딕셔너리 형태로 두 값을 동시에 흘려보내면 라우팅 함수에서 `info["language"]`와 `info["text"]`를 모두 활용할 수 있어요

In [37]:
# ---------------------------------------------------
# 1단계: 언어 감지 체인 구성
# - LLM에게 언어를 판별하게 하고 한 단어(한국어/영어/기타)만 반환하도록 해요
# ---------------------------------------------------

# ============================================================
# TODO: 언어 감지 체인을 구성하세요
# 힌트: PromptTemplate.from_template()으로 언어 분류 프롬프트를 작성하고
#       model | StrOutputParser()로 연결하세요
# 예상 결과: detect_chain.invoke("안녕하세요") → "한국어"
# ============================================================

# 여기에 코드를 작성하세요
prompt = PromptTemplate.from_template(
  """
  아래 입력 글자를 읽고 무슨 언어인지 확인하고 반환하세요.
  {text}

  예 - 한국어
  결과:
  """
)

detect_chain = (
  prompt
  | model
  | StrOutputParser()
)

언어 감지가 잘 동작하는지 확인했어요. 이제 각 언어에 대응하는 번역 체인 3개를 만들고, 라우팅 함수로 연결해볼게요.

In [38]:
# ---------------------------------------------------
# 2단계: 번역 체인 3개 구성
# - 한국어 → 영어, 영어 → 한국어, 기타 → 한국어
# ---------------------------------------------------

# ============================================================
# TODO: 3개의 번역 체인을 구성하세요
# 힌트: ChatPromptTemplate.from_template()에 {text} 변수를 사용하고
#       각 체인을 model | StrOutputParser()로 마무리해요
# 예상 결과: ko_to_en_chain.invoke({"text": "안녕하세요"}) → "Hello"
# ============================================================

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 {from_lang} 언어를 {to_lang}로 번역하시오"),
    ("human", "{text}")
])

ko_to_en_translate_prompt = translate_prompt.partial(from_lang="한국어" ,to_lang="영어")
en_to_ko_translate_prompt = translate_prompt.partial(from_lang="영어", to_lang="한국어")
etc_to_ko_translate_prompt = translate_prompt.partial(from_lang="기타", to_lang="한국어")

ko_to_en_chain = (
    # {
    #     "language": detect_chain,
    #     "text": itemgetter("text")
    # }
    # RunnablePassthrough.assign(language=detect_chain)
    ko_to_en_translate_prompt
    | model
    | StrOutputParser()
)

en_to_ko_chain = (
    # RunnablePassthrough.assign(language=detect_chain)
    en_to_ko_translate_prompt
    | model
    | StrOutputParser()
)

etc_to_ko_chain = (
    # RunnablePassthrough.assign(language=detect_chain)
    etc_to_ko_translate_prompt
    | model
    | StrOutputParser()
)

translate_chain = (
    
)


res = ko_to_en_chain.invoke({"text": "이건 영어야"})

res



'This is English.'

In [48]:
# ---------------------------------------------------
# 3단계: 라우팅 함수 + 전체 파이프라인 구성
# - route() 함수: 감지된 언어에 따라 적절한 번역 체인을 선택해요
# - RunnableLambda로 래핑해서 파이프라인에 연결해요
# ---------------------------------------------------

# ============================================================
# TODO: 라우팅 함수와 전체 체인을 구성하세요
# 힌트: route(info)에서 info["language"]로 언어를 확인하고
#       해당 번역 체인을 반환해요. 반환된 체인은 info["text"]를 입력으로 받아요
# 예상 결과: router_chain.invoke("안녕하세요") → 영어 번역 결과
# ============================================================

# 여기에 코드를 작성하세요
def route(info: dict):
    from_lang = info["from_lang"]

    # if "한국어" in from_lang:
    #     return translate_prompt.partial(to_lang="영어") | model | StrOutputParser()
    # elif "영어" in from_lang:
    #     return translate_prompt.partial(to_lang="한국어") | model | StrOutputParser()
    # else:
    #     return translate_prompt.partial(to_lang="한국어") | model | StrOutputParser()
    if "한국어" in from_lang:
        return ko_to_en_chain
    elif "영어" in from_lang:
        return en_to_ko_chain
    else:
        return etc_to_ko_chain

router_chain = (
    {
        "text": RunnablePassthrough()
    }
    | RunnablePassthrough.assign(from_lang=detect_chain)
    # | RunnableLambda(route)
    | RunnableLambda(route)
)

res = router_chain.invoke("午餐吃得好吗？")
res


'점심 맛있게 드셨나요?'

In [49]:
# ---------------------------------------------------
# 라우터 체인 테스트
# - 3가지 언어 입력으로 라우팅이 올바르게 동작하는지 확인해요
# ---------------------------------------------------

test_cases = [
    "안녕하세요. 오늘 날씨가 맑고 좋네요.",
    "The quick brown fox jumps over the lazy dog.",
    "Bonjour! Comment allez-vous aujourd'hui?",
]

for text in test_cases:
    print(f"입력: {text}")
    result = router_chain.invoke(text)
    print(f"결과: {result}")
    print("-" * 60)

입력: 안녕하세요. 오늘 날씨가 맑고 좋네요.
결과: Hello. The weather is clear and nice today.
------------------------------------------------------------
입력: The quick brown fox jumps over the lazy dog.
결과: 빠른 갈색 여우가 게으른 개를 뛰어넘는다.
------------------------------------------------------------
입력: Bonjour! Comment allez-vous aujourd'hui?
결과: 안녕하세요! 오늘 기분이 어떠신가요?
------------------------------------------------------------


---

## 실습 2: 병렬 분석 파이프라인

실습 1에서는 입력에 따라 하나의 체인을 선택해서 실행했어요.<br/>
실습 2에서는 정반대로, **하나의 텍스트를 3개의 체인이 동시에 분석**하도록 구성해요.<br/>
`RunnableParallel`(병렬 실행 가능 객체)을 활용하면 API 호출 3번을 거의 동시에 처리할 수 있어요.

아래 다이어그램은 병렬 분석 파이프라인의 구조를 보여줘요.

```mermaid
graph LR
    A["분석 대상 텍스트"]:::input
    A -->|"동시 실행"| B["감정 분석 체인"]:::process
    A -->|"동시 실행"| C["키워드 추출 체인"]:::process
    A -->|"동시 실행"| D["요약 체인"]:::process
    B --> E["결과 병합 함수"]:::process
    C --> E
    D --> E
    E --> F["통합 분석 보고서"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

### 실습 2 과제

**과제**: 감정 분석, 키워드 추출, 요약 체인 3개를 `RunnableParallel`로 병렬 실행하고, 결과를 하나의 보고서 문자열로 합치세요.

**요구사항**:
- 3개의 분석 체인이 `RunnableParallel`로 병렬 실행되어야 해요
- `RunnableLambda`로 3개 결과를 받아 하나의 보고서 문자열로 합쳐요
- 최종 결과는 사람이 읽기 좋은 형태로 출력돼야 해요

**힌트**: `RunnableParallel`의 출력은 딕셔너리예요. 예: `{"sentiment": "긍정", "keywords": "...", "summary": "..."}`

In [69]:
# ---------------------------------------------------
# 1단계: 3개 분석 체인 정의
# - 감정 분석 / 키워드 추출 / 요약 체인을 각각 만들어요
# ---------------------------------------------------

# ============================================================
# TODO: 감정 분석, 키워드 추출, 요약 체인을 각각 구성하세요
# 힌트: ChatPromptTemplate.from_template()에 {text} 변수를 사용하고
#       각 체인은 model | StrOutputParser()로 마무리해요
# ============================================================

# 여기에 코드를 작성하세요
sentiment_prompt = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트의 감정을 분석하고 어떤 감정인지 단답형으로 답변하시오"),
  ("human", "{text}"),
])

keyword_prompt = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트의 키워드를 여러개 추출하고 컴마(,) 로 구분해서 반환하시오"),
  ("human", "{text}"),
])

summary_prompt = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 요약하고 요약한 내용을 반환하시오"),
  ("human", "{text}"),
])


sentiment_chain = (
  sentiment_prompt
  | model
  | StrOutputParser()
)

keyword_chain = (
  keyword_prompt
  | model
  | StrOutputParser()
)

summary_chain = (
  summary_prompt
  | model
  | StrOutputParser()
)

res = sentiment_chain.invoke({"text": "아 배고프다!"})
# res = keyword_chain.invoke({"text": "아 배고프다!"})
# res = summary_chain.invoke({"text": "아 배고프다!"})
res

'배고픔'

In [70]:
# ---------------------------------------------------
# 2단계: RunnableParallel로 병렬 실행 구성
# - 3개 체인을 동시에 실행해 딕셔너리로 결과를 수집해요
# ---------------------------------------------------

# ============================================================
# TODO: RunnableParallel로 3개 체인을 병렬 실행하도록 구성하세요
# 힌트: RunnableParallel(sentiment=sentiment_chain, keywords=keyword_chain, summary=summary_chain)
# 예상 결과: {"sentiment": "...", "keywords": "...", "summary": "..."}
# ============================================================

# 여기에 코드를 작성하세요
parallel = RunnableParallel(sentiment=sentiment_chain, keywords=keyword_chain, summary=summary_chain)
res = parallel.invoke({"text": (
  "아 배고프다!"
  "오늘 뭐 먹지?"
  )})
res

{'sentiment': '배고픔',
 'keywords': '배고프다, 오늘, 뭐, 먹지',
 'summary': '배고프다는 표현과 함께 오늘 먹을 음식을 고민하고 있는 상황입니다.'}

In [74]:
# ---------------------------------------------------
# 3단계: 결과 병합 함수 + 전체 파이프라인 연결
# - 딕셔너리 형태의 분석 결과를 사람이 읽기 좋은 보고서로 합쳐요
# ---------------------------------------------------

# ============================================================
# TODO: merge_results 함수를 구성하고 전체 파이프라인을 완성하세요
# 힌트: results["sentiment"], results["keywords"], results["summary"]로 각 결과에 접근해요
# 예상 결과: 보고서 형식의 문자열
# ============================================================

# 여기에 코드를 작성하세요
def merge_results(results: dict):
    sentiment = results["sentiment"]
    keywords = results["keywords"]
    summary = results["summary"]

    report = f"""
    통합 분석 보고서
    ------------------------------------------------------------
    1. 감정 분석 결과:
    {sentiment}
    2. 주요 키워드:
    {keywords}
    3. 요약 내용:
    {summary}
    """
    return report.strip()

parallel_analysis = RunnableParallel(
    sentiment=emotion_chain,
    keywords=keyword_chain,
    summary=summary_chain
)

analysis_pipeline = (
    parallel_analysis 
    | RunnableLambda(merge_results)
)


In [73]:
# ---------------------------------------------------
# 병렬 분석 파이프라인 테스트
# ---------------------------------------------------

sample_text = {
    "text": (
        "인공지능 기술의 발전으로 의료 분야에서 암 조기 진단 정확도가 크게 높아지고 있어요. "
        "특히 영상 판독 분야에서 AI가 전문의 수준의 성능을 보이면서 "
        "환자들이 더 빠르고 정확한 진단을 받을 수 있게 됐어요. "
        "다만 최종 판단은 여전히 의사가 내려야 한다는 점에서 "
        "AI는 의사를 보조하는 역할을 담당하고 있어요."
    )
}

report = analysis_pipeline.invoke(sample_text)
print(report)

[ 📊 통합 분석 보고서 ]
    ------------------------------------------------------------
    1. 감정 분석 결과:
    긍정적
    2. 주요 키워드:
    인공지능, 의료 분야, 암 조기 진단, 영상 판독, AI, 전문의, 성능, 환자, 빠르고 정확한 진단, 최종 판단, 의사, 보조 역할
    3. 요약 내용:
    인공지능 기술의 발전으로 의료 분야에서 암 조기 진단의 정확도가 향상되고 있으며, 특히 영상 판독에서 AI가 전문의 수준의 성능을 보여 환자들에게 빠르고 정확한 진단을 제공하고 있습니다. 그러나 최종 판단은 여전히 의사가 내려야 하며, AI는 의사를 보조하는 역할을 하고 있습니다.


---

## 실습 3: 장애 대응 체인 (Fallbacks)

실제 서비스에서는 API 장애, 잘못된 모델 이름, 네트워크 오류 등 다양한 상황이 발생해요.<br/>
`with_fallbacks()`(폴백 설정 메서드)를 사용하면 **기본 체인이 실패할 때 자동으로 대체 로직으로 전환**할 수 있어요.<br/>
이번 실습에서는 의도적으로 오류를 발생시켜서 폴백 동작을 직접 확인해볼게요.

아래 다이어그램은 장애 대응 흐름을 보여줘요.

```mermaid
graph TD
    A["질문 입력"]:::input
    A --> B["Primary 체인<br/>(gpt-4o-mini)"]:::process
    B -->|"성공"| C["응답 반환"]:::output
    B -->|"실패 (오류)"| D["Fallback 체인<br/>(규칙 기반 응답)"]:::error
    D -->|"폴백 응답"| C

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
```

### 실습 3 과제

**과제**: Primary 체인(gpt-4o-mini)과 Fallback 체인(규칙 기반)을 구성하고, `with_fallbacks()`로 연결한 뒤 의도적 오류로 폴백 동작을 검증하세요.

**요구사항**:
- Primary 체인: 정상 gpt-4o-mini 체인 (max_retries=0으로 즉시 실패하도록 설정)
- Fallback 체인: LLM 없이 규칙 기반으로 응답하는 `RunnableLambda` 체인
- 오류 없이 폴백이 투명하게 동작하는지 확인해요

**힌트**: `unittest.mock.patch`로 API 호출을 가로채서 의도적으로 오류를 발생시킬 수 있어요

In [76]:
# ---------------------------------------------------
# 1단계: Primary 체인 구성
# - 정상 상태에서는 gpt-4o-mini가 응답해요
# - max_retries=0: 오류 발생 시 재시도 없이 즉시 예외를 던져요
# ---------------------------------------------------

# ============================================================
# TODO: Primary 체인을 구성하세요
# 힌트: ChatOpenAI(model="gpt-4o-mini", max_retries=0)으로 모델을 초기화하고
#       프롬프트 | 모델 | StrOutputParser() 순서로 체인을 완성하세요
# ============================================================

# 여기에 코드를 작성하세요

primary_prompt = ChatPromptTemplate.from_template(
  """
  다음 문장을 파악하시오

  {text}
  """
)
primary_model = ChatOpenAI(model="gpt-4o-mini", max_retries=0)

primary_chain = (
  primary_prompt
  | primary_model
  | StrOutputParser()
)

res = primary_chain.invoke({"text": "안녕하세요"})

res

'안녕하세요! 어떻게 도와드릴까요?'

In [ ]:
# ---------------------------------------------------
# 2단계: Fallback 체인 구성 (규칙 기반 응답)
# - LLM 없이 키워드를 기반으로 미리 준비한 응답을 반환해요
# - 실제 서비스라면 캐시나 디폴트 메시지가 이 역할을 해요
# ---------------------------------------------------

# ============================================================
# TODO: 규칙 기반 Fallback 체인을 구성하세요
# 힌트: RunnableLambda를 사용해서 질문 키워드를 검사하고
#       미리 준비한 응답을 반환하는 함수를 만들어요
# 예상 결과: 키워드가 있으면 준비된 답변, 없으면 기본 메시지
# ============================================================

# 여기에 코드를 작성하세요


In [ ]:
# ---------------------------------------------------
# 3단계: with_fallbacks()로 장애 대응 체인 완성
# - primary_chain이 실패하면 자동으로 fallback_chain이 실행돼요
# ---------------------------------------------------

# ============================================================
# TODO: with_fallbacks()로 primary_chain과 fallback_chain을 연결하세요
# 힌트: chain_with_fallback = primary_chain.with_fallbacks([fallback_chain])
# ============================================================

# 여기에 코드를 작성하세요
pass

In [ ]:
# ---------------------------------------------------
# 정상 동작 확인: Primary 체인이 응답해요
# ---------------------------------------------------

print("[정상 동작 테스트]")
normal_result = chain_with_fallback.invoke({"question": "LangChain이 무엇인가요?"})
print(f"결과: {normal_result}")

In [ ]:
# ---------------------------------------------------
# 장애 상황 시뮬레이션: Primary 체인이 오류를 발생시켜요
# - unittest.mock.patch로 OpenAI API 호출을 가로채서 RateLimitError를 강제로 발생시켜요
# - 이때 chain_with_fallback은 자동으로 fallback_chain을 실행해요
# ---------------------------------------------------

import httpx
from unittest.mock import patch
from openai import RateLimitError

# 의도적 RateLimitError 객체 생성
mock_request = httpx.Request("GET", "/")
mock_response = httpx.Response(429, request=mock_request)
rate_limit_error = RateLimitError("rate limit exceeded", response=mock_response, body="")

print("[장애 상황 시뮬레이션 테스트]")
# OpenAI API 호출을 가로채서 RateLimitError를 발생시켜요
with patch("openai.resources.chat.completions.Completions.create", side_effect=rate_limit_error):
    fallback_result = chain_with_fallback.invoke({"question": "LangChain이 무엇인가요?"})
    print(f"결과: {fallback_result}")

print("\n→ Primary가 실패해도 Fallback이 자동으로 응답했어요!")

In [ ]:
# ---------------------------------------------------
# 다양한 키워드로 폴백 규칙 기반 응답 확인
# ---------------------------------------------------

test_questions = ["오늘 날씨가 어때요?", "지금 시간이 몇 시예요?", "파이썬 문법을 알려주세요"]

print("[폴백 규칙 기반 응답 확인]")
with patch("openai.resources.chat.completions.Completions.create", side_effect=rate_limit_error):
    for q in test_questions:
        res = chain_with_fallback.invoke({"question": q})
        print(f"Q: {q}")
        print(f"A: {res}")
        print("-" * 50)

---

## 실습 4: 맞춤형 데이터 변환 파이프라인

지금까지 라우팅, 병렬 처리, 장애 대응을 배웠어요.<br/>
마지막 실습에서는 Ch05에서 배운 모든 패턴을 결합해서 **5단계 데이터 변환 파이프라인**을 구성해요.<br/>
`RunnablePassthrough.assign()`으로 데이터를 보강하고, `RunnableLambda`로 변환하고, `@chain` 데코레이터로 전체를 깔끔하게 감싸요.

아래 다이어그램은 5단계 파이프라인 전체 흐름을 보여줘요.

```mermaid
graph LR
    A["1단계<br/>원본 입력"]:::input
    A -->|"분류"| B["2단계<br/>텍스트 분류<br/>(뉴스/리뷰/기타)"]:::process
    B -->|"메타데이터 보강"| C["3단계<br/>assign으로<br/>카테고리 추가"]:::process
    C -->|"LLM 처리"| D["4단계<br/>카테고리별<br/>LLM 분석"]:::process
    D -->|"후처리"| E["5단계<br/>Lambda로<br/>포맷 정리"]:::process
    E --> F["최종 출력"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

### 실습 4 과제

**과제**: 입력 텍스트를 분류, 보강, LLM 분석, 후처리하는 5단계 파이프라인을 `@chain` 데코레이터로 작성하세요.

**요구사항**:
- 1단계: 원본 텍스트 입력
- 2단계: LLM으로 텍스트를 `뉴스`, `리뷰`, `기타` 중 하나로 분류
- 3단계: `RunnablePassthrough.assign()`으로 분류 결과를 원본 데이터에 추가
- 4단계: 분류 카테고리에 맞는 LLM 분석 실행
- 5단계: `RunnableLambda`로 최종 결과를 보기 좋게 포맷팅
- 전체 흐름을 `@chain` 데코레이터로 하나의 Runnable 객체로 감싸요

**힌트**: `@chain` 데코레이터를 사용하면 `invoke()` 메서드가 자동으로 생기고 LangSmith에서 추적도 돼요

In [77]:
# ---------------------------------------------------
# 1단계 + 2단계: 텍스트 분류 체인
# - 입력 텍스트가 뉴스/리뷰/기타 중 어디에 속하는지 판별해요
# ---------------------------------------------------

# ============================================================
# TODO: 텍스트 분류 체인을 구성하세요
# 힌트: 프롬프트에서 '뉴스', '리뷰', '기타' 중 하나만 반환하도록 지시해요
# 예상 결과: classify_chain.invoke({"text": "주가가 10% 급등했다"}) → "뉴스"
# ============================================================

# 여기에 코드를 작성하세요
classify_prompt = ChatPromptTemplate.from_template("""
다음 텍스트를 확인하고 '뉴스', '리뷰', '기타' 중 어울리는 카테고리를 선택해서 반환하시오

{text}
""")

classify_chain = (
    classify_prompt
    | model
    | StrOutputParser()
)

res = classify_chain.invoke({"text": "한국은행이 기준금리를 0.25%포인트 인상해서 연 3.75%로 결정했어요"})
res

'뉴스'

In [79]:
# ---------------------------------------------------
# 4단계: 카테고리별 LLM 분석 체인
# - 분류 결과에 따라 다른 관점으로 텍스트를 분석해요
# ---------------------------------------------------

# ============================================================
# TODO: 뉴스/리뷰/기타 각 카테고리에 맞는 분석 체인을 구성하세요
# 힌트: 각 체인은 {text}와 {category} 변수를 받고
#       카테고리 특성에 맞는 분석 관점을 프롬프트에 반영해요
# ============================================================

# 여기에 코드를 작성하세요
news_prompt = ChatPromptTemplate.from_template("""
    다음 텍스트는 뉴스입니다.
    뉴스에 맞는 분석을 하고 분석한 내용을 반환하시오

    텍스트:
    {text}
""")

review_prompt = ChatPromptTemplate.from_template("""
    다음 텍스트는 리뷰입니다.
    리뷰에 맞는 분석을 하고 분석한 내용을 반환하시오

    텍스트:
    {text}
""")

etc_prompt = ChatPromptTemplate.from_template("""
    다음 텍스트는 기준이 정해지지 않은 텍스트 입니다.
    자유롭게 분석하고 분석한 내용을 반환하시오

    텍스트:
    {text}
""")

news_chain = (
    news_prompt
    | model
    | StrOutputParser()
)

review_chain = (
    review_prompt
    | model
    | StrOutputParser()
)

etc_chain = (
    etc_prompt
    | model
    | StrOutputParser()
)

res = news_chain.invoke({"text": "한국은행이 기준금리를 0.25%포인트 인상해서 연 3.75%로 결정했어요"})
res


'분석 내용:\n\n1. **금리 인상 배경**: 한국은행이 기준금리를 0.25%포인트 인상한 것은 경제 상황에 대한 반응으로 볼 수 있습니다. 일반적으로 금리 인상은 인플레이션 압력이나 경제 성장률 상승에 대응하기 위해 이루어집니다. 따라서, 이번 결정은 한국 경제의 회복세나 물가 상승 우려가 반영된 것으로 해석될 수 있습니다.\n\n2. **금리 인상의 영향**: 기준금리가 3.75%로 인상됨에 따라, 대출 금리와 예금 금리가 상승할 가능성이 큽니다. 이는 가계와 기업의 금융 부담을 증가시킬 수 있으며, 소비와 투자에 부정적인 영향을 미칠 수 있습니다. 반면, 예금자에게는 더 높은 이자를 제공하여 저축을 장려할 수 있습니다.\n\n3. **경제 전망**: 한국은행의 금리 인상은 향후 경제 성장률과 물가 상승률에 대한 전망을 반영합니다. 만약 경제가 지속적으로 성장하고 물가가 안정적으로 유지된다면, 추가적인 금리 인상이 있을 수 있습니다. 반대로, 경제가 둔화되거나 물가 상승이 억제된다면 금리 인상이 중단되거나 인하될 가능성도 존재합니다.\n\n4. **시장 반응**: 금리 인상 소식은 금융 시장에 즉각적인 영향을 미칠 수 있습니다. 주식 시장은 금리 인상에 민감하게 반응할 수 있으며, 특히 금리가 높은 환경에서는 성장주보다 가치주가 선호될 가능성이 있습니다. 또한, 외환 시장에서도 원화의 가치에 영향을 미칠 수 있습니다.\n\n결론적으로, 한국은행의 기준금리 인상은 경제 전반에 걸쳐 다양한 영향을 미칠 것으로 예상되며, 이는 소비자와 기업의 행동, 금융 시장의 움직임, 그리고 향후 경제 정책에 중요한 신호로 작용할 것입니다.'

In [82]:
# ---------------------------------------------------
# @chain 데코레이터로 전체 5단계 파이프라인 구성
# - @chain: 일반 함수를 Runnable 객체로 변환해서 invoke()를 사용할 수 있게 해요
# - RunnablePassthrough.assign(): 원본 딕셔너리에 새 키를 추가해요
# - RunnableLambda: 중간 변환 함수를 Runnable로 감싸요
# ---------------------------------------------------

# ============================================================
# TODO: @chain 데코레이터로 5단계 파이프라인을 완성하세요
# 힌트:
#   - 2단계: classify_chain.invoke()로 분류
#   - 3단계: RunnablePassthrough.assign(category=...)으로 보강
#   - 4단계: category 값에 따라 분석 체인 선택 후 실행
#   - 5단계: 결과를 보기 좋게 포맷팅
# ============================================================

# 여기에 코드를 작성하세요
@chain
def smart_transform_pipeline(input_text: str):
    """
    5단계 파이프라인 구성
    """
    info = {"text": input_text}
    category = classify_chain.invoke(text)

    if "뉴스" in category:
        res = news_chain.invoke(info)
    elif "리뷰" in category:
        res = review_chain.invoke(info)
    else:
        res = etc_chain.invoke(info)

    final_report = f"""
    [ 텍스트 분석 통합 리포트 ]
    ------------------------------------------------------------
    - 텍스트 유형: {category}
    - 분류 근거: 원본 텍스트 내용을 바탕으로 LLM이 판별함
    ------------------------------------------------------------
    [ 분석 상세 내용 ]
    {res}
    ------------------------------------------------------------
    """

    return final_report


In [83]:
# ---------------------------------------------------
# 5단계 파이프라인 테스트
# - 3가지 유형의 텍스트로 전체 파이프라인 동작을 확인해요
# ---------------------------------------------------

test_inputs = [
    "한국은행이 기준금리를 0.25%포인트 인상해서 연 3.75%로 결정했어요.",
    "이 스마트폰은 카메라 성능이 탁월하지만 배터리 지속 시간이 아쉬워요.",
    "오늘 저녁에 친구들과 보드게임을 하려고 해요.",
]

for input_text in test_inputs:
    print("=" * 60)
    # @chain 데코레이터 덕분에 invoke()를 바로 호출할 수 있어요
    result = smart_transform_pipeline.invoke(input_text)
    print(result)
    print()


    [ 텍스트 분석 통합 리포트 ]
    ------------------------------------------------------------
    - 텍스트 유형: '기타'
    - 분류 근거: 원본 텍스트 내용을 바탕으로 LLM이 판별함
    ------------------------------------------------------------
    [ 분석 상세 내용 ]
    이 텍스트는 한국은행의 기준금리 인상에 대한 정보를 담고 있습니다. 다음은 이 텍스트에 대한 분석 내용입니다.

1. **주제**: 한국은행의 기준금리 인상
   - 텍스트는 한국은행이 기준금리를 인상했다는 사실을 중심으로 하고 있습니다. 이는 경제 정책과 관련된 중요한 결정입니다.

2. **수치적 정보**: 
   - 기준금리 인상폭: 0.25%포인트
   - 새로운 기준금리: 연 3.75%
   - 이러한 수치는 금융 시장과 경제 전반에 영향을 미칠 수 있는 중요한 요소입니다.

3. **경제적 맥락**:
   - 기준금리는 중앙은행이 설정하는 금리로, 이는 대출과 예금의 이자율에 영향을 미치며, 소비와 투자에 직접적인 영향을 줄 수 있습니다.
   - 금리 인상은 일반적으로 인플레이션 억제, 경제 과열 방지 등의 목적을 가지고 시행됩니다.

4. **정책적 의도**:
   - 한국은행이 금리를 인상한 이유는 경제 상황에 따라 다를 수 있지만, 일반적으로는 물가 상승률을 조절하거나 경제의 과열을 방지하기 위한 조치로 해석될 수 있습니다.

5. **시장 반응**:
   - 금리 인상 소식은 금융 시장에서 주식, 채권, 외환 등 다양한 자산의 가격에 영향을 미칠 수 있습니다. 투자자들은 이러한 변화를 반영하여 포트폴리오를 조정할 수 있습니다.

6. **사회적 영향**:
   - 개인 대출자와 기업 모두 금리 인상의 영향을 받을 수 있습니다. 대출 이자 부담이 증가할 수 있으며, 이는 소비와 투자에 부정적인 영향을 미칠 수 있습니다.

결론적으로, 이

> **실무 팁**: `@chain` 데코레이터로 감싼 함수는 LangSmith에서 단일 추적 단위로 기록돼요. 복잡한 파이프라인의 실행 흐름을 한눈에 파악하고 디버깅할 때 큰 도움이 돼요.

---

## 정리

이번 실습에서 배운 내용을 정리해볼게요:

- **실습 1 (다국어 라우터)**: `RunnableLambda`와 `route()` 패턴으로 LLM이 감지한 언어에 따라 서로 다른 번역 체인으로 동적 분기하는 방법을 익혔어요
- **실습 2 (병렬 분석)**: `RunnableParallel`로 감정 분석·키워드 추출·요약을 동시에 수행하고, `RunnableLambda`로 결과를 하나의 보고서로 합치는 방법을 익혔어요
- **실습 3 (장애 대응)**: `with_fallbacks()`로 API 장애 시 규칙 기반 응답으로 자동 전환하는 체인을 구성하고, 의도적 오류로 동작을 검증했어요
- **실습 4 (데이터 변환)**: `RunnablePassthrough.assign()`으로 데이터를 보강하고, `@chain` 데코레이터로 5단계 파이프라인을 하나의 Runnable 객체로 감싸는 방법을 익혔어요

다음 노트북에서는 RAG(Retrieval-Augmented Generation, 검색 증강 생성) 파이프라인을 다룰 거예요. 오늘 배운 LCEL 고급 패턴이 RAG 체인 구성에서도 핵심적으로 활용되므로, 이번 실습의 패턴을 잘 기억해 두세요.